In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import re
import urllib.request

# Function to read and process word count data from Project Gutenberg
# only for #5
def read_word_counts(book_title):
    if book_title == 'compte':  # Le Compte de Monte Cristo (in French)
        url, char_start, char_end = "https://www.gutenberg.org/cache/epub/17989/pg17989.txt", 313, 13187
    else: 
        url, char_start, char_end = "https://www.gutenberg.org/cache/epub/4300/pg4300.txt", 198, 265180
 
    # Fetch the content
    response = urllib.request.urlopen(url)
    content = response.read().decode('utf-8')  # Decode the bytes to string
    
    words = content.split()
    
    words = words[char_start:char_end]  # Remove the Project Gutenberg header
    # print first and last 10 words
    print(words[:10])
    print(words[-10:])
    print(f"Number of words in the book: {len(words)}")
 
    # Count occurrences of each word
    word_counts = pd.Series(words).value_counts().reset_index()
    word_counts.columns = ['Word', 'Count']
    return word_counts
 
# Function to estimate innovation rate
def estimate_innovation_rate(counts_df, book_title=''):
    if book_title == 'u':
        # get data from https://pdodds.w3.uvm.edu/teaching/courses/2024-2025pocsverse/docs/ulysses.txt
        url = "https://pdodds.w3.uvm.edu/teaching/courses/2024-2025pocsverse/docs/ulysses.txt"
        response = urllib.request.urlopen(url)
        content = response.read().decode('utf-8')
        # document is formatted as token:count
        # create counts_df
        words = content.split()
        words = words[1:]
        words = [word.split(':') for word in words]
        counts_df = pd.DataFrame(words, columns=['Word', 'Count'])

    counts_df = counts_df.sort_values(by='Count', ascending=False).reset_index(drop=True)
    counts_df['Rank'] = np.arange(1, len(counts_df) + 1)
    total_words = counts_df['Count'].sum()
    distinct_words = counts_df[counts_df['Count'] == 1]['Count'].sum()
    print(f"distinct_words: {distinct_words}")
    rho_est = distinct_words / total_words
 
    return rho_est, counts_df
 
# Function to calculate theoretical and empirical values
def calculate_theoretical_and_empirical(counts_df, rho_est):
    n_1 = 1 / (2 - rho_est)
    n_2 = n_1 * (1 - rho_est) / (1 + (1 - rho_est) * 2)
    n_3 = n_2 * (2 - rho_est) / (2 + (2 - rho_est) * 3)    
 
    return n_1, n_2, n_3


 
# Main execution
book_title = 'u'  # Set the book title
#book_title = 'pride'
#book_title = 'conpte'
word_counts_df = read_word_counts(book_title)
innovation_rate, processed_counts_df = estimate_innovation_rate(word_counts_df, 'u')
 
 
 
 
# Calculate theoretical and empirical values. this is based on unique/total words given the data where that is innovation rate
theoretical_values_est = calculate_theoretical_and_empirical(processed_counts_df, innovation_rate)
 
 
#our_rho
 
if(book_title =='u'):
    simon_rho=0.115
else:
    our_rho=0.11861461394906046
    
theoretical_values_est_Our_Rho= calculate_theoretical_and_empirical(processed_counts_df, our_rho)
 
 
 
# Output results
print(f"Estimated Innovation Rate (rho): {innovation_rate:.4f}")
 
print(f"Theoretical n_1: {theoretical_values_est[0]:.3f}, n_2: {theoretical_values_est[1]:.3f}, n_3: {theoretical_values_est[2]:.3f}")
 
#ours
print(f"Estimated Innovation Rate (rho) based on Simon: {our_rho:.4f}")
print(f"Theoretical n_1: {theoretical_values_est_Our_Rho[0]:.3f}, n_2: {theoretical_values_est_Our_Rho[1]:.3f}, n_3: {theoretical_values_est_Our_Rho[2]:.3f}")
 

['Stately,', 'plump', 'Buck', 'Mulligan', 'came', 'from', 'the', 'stairhead,', 'bearing', 'a']
['like', 'mad', 'and', 'yes', 'I', 'said', 'yes', 'I', 'will', 'Yes.']
Number of words in the book: 264982


TypeError: can only concatenate str (not "int") to str

In [10]:
import pandas as pd
import numpy as np
import urllib.request

# Function to read and process word count data from Project Gutenberg
def read_word_counts(book_title):
    if book_title == 'compte':  # Le Compte de Monte Cristo (in French)
        url, char_start, char_end = "https://www.gutenberg.org/cache/epub/17989/pg17989.txt", 313, 131872
    elif book_title == 'pride':  # Pride and Prejudice
        url, char_start, char_end = "https://www.gutenberg.org/cache/epub/42671/pg42671.txt", 337, 128319

    # Fetch the content
    response = urllib.request.urlopen(url)
    content = response.read().decode('utf-8')  # Decode the bytes to string
    
    words = content.split()
    
    words = words[char_start:char_end]  # Remove the Project Gutenberg header/footer
    # print(f"Number of words in {book_title}: {len(words)}")
 
    # Count occurrences of each word
    word_counts = pd.Series(words).value_counts().reset_index()
    word_counts.columns = ['Word', 'Count']
    return word_counts

# Function to estimate innovation rate
def estimate_innovation_rate(counts_df):
    counts_df = counts_df.sort_values(by='Count', ascending=False).reset_index(drop=True)
    counts_df['Rank'] = np.arange(1, len(counts_df) + 1)
    total_words = counts_df['Count'].sum()
    distinct_words = counts_df[counts_df['Count'] == 1]['Count'].sum()
    rho_est = distinct_words / total_words
    
    return rho_est, counts_df

# Function to calculate theoretical and empirical values
def calculate_theoretical_and_empirical(rho_est):
    n_1 = 1 / (2 - rho_est)
    n_2 = n_1 * (1 - rho_est) / (1 + (1 - rho_est) * 2)
    n_3 = n_2 * (2 - rho_est) / (2 + (2 - rho_est) * 3)    
    return n_1, n_2, n_3

# Main execution function for each book
def analyze_book(book_title):
    word_counts_df = read_word_counts(book_title)
    innovation_rate, processed_counts_df = estimate_innovation_rate(word_counts_df)
    
    # Calculate theoretical and empirical values
    theoretical_values_est = calculate_theoretical_and_empirical(innovation_rate)
    
    print(f"\nBook: {book_title}")
    print(f"Estimated Innovation Rate (rho): {innovation_rate:.4f}")
    print(f"Theoretical n_1: {theoretical_values_est[0]:.3f}, n_2: {theoretical_values_est[1]:.3f}, n_3: {theoretical_values_est[2]:.3f}")
    empirical_values = 
    return innovation_rate, theoretical_values_est

# Ulysses hardcoded edge case
def analyze_ulysses():
    ulysses_innovation_rate = 0.11861461394906046  # Hardcoded for Ulysses
    ulysses_values = calculate_theoretical_and_empirical(ulysses_innovation_rate)
    print(f"\nBook: Ulysses")
    print(f"Derived Innovation Rate: {ulysses_innovation_rate:.4f}")
    print(f"Empirical n_1: {ulysses_values[0]:.3f}, n_2: {ulysses_values[1]:.3f}, n_3: {ulysses_values[2]:.3f}")
    print(f"Simon's Rate: 0.1150")
    empirical_ulysses_values = calculate_theoretical_and_empirical(0.1150)
    print(f"Theoretical n_1: {empirical_ulysses_values[0]:.3f}, n_2: {empirical_ulysses_values[1]:.3f}, n_3: {empirical_ulysses_values[2]:.3f}")



# Analyze all three books
analyze_ulysses()  # Ulysses with hardcoded values
analyze_book('compte')  # Le Compte de Monte Cristo
analyze_book('pride')   # Pride and Prejudice



Book: Ulysses
Derived Innovation Rate: 0.1186
Empirical n_1: 0.532, n_2: 0.170, n_3: 0.042
Simon's Rate: 0.1150
Theoretical n_1: 0.531, n_2: 0.169, n_3: 0.042

Book: compte
Estimated Innovation Rate (rho): 0.1049
Theoretical n_1: 0.528, n_2: 0.169, n_3: 0.042

Book: pride
Estimated Innovation Rate (rho): 0.0599
Theoretical n_1: 0.515, n_2: 0.168, n_3: 0.042


(np.float64(0.05988115191223526),
 (np.float64(0.5154323411607634),
  np.float64(0.16823877400211132),
  np.float64(0.04173763876023153)))

In [ ]:
import pandas as pd
import numpy as np
 
# Function to read and process word count data from Project Gutenberg
def read_word_counts(book_title):
  
    if book_title=="u":
        file_path = "/Users/dannysatterthwaite/Desktop/School-UVM/Fall Senior/CS-2100/pocs-6-4.rtf"
        
        # Split so we have words
        data = [line.split(':') for line in content.strip().split('\n') if ':' in line]
        df = pd.DataFrame(data, columns=["Word", "Count"])
        
        
        # # Cleaning
        df['Word'] = df['Word'].str.replace(r'\\', '', regex=True)
        df['Count'] = df['Count'].str.replace(r'\\', '', regex=True)
        df['Count'] = df['Count'].str.replace(r'[^\d]', '', regex=True)
 
        # Remove any rows where 'Count' is still empty
        df = df[df['Count'].str.strip() != '']
 
        # Convert count to int
        df["Count"] = df["Count"].astype(int)
        
        
        
        
    elif book_title=="pp":
        file_path = "/Users/dannysatterthwaite/Downloads/pride and pred pocs hw 6-5.rtf"
        
                # read it in
        with open(file_path, 'r') as file:
            raw_text = file.read()
 
 
 
        # strip all the bad characters and new line characters.
        cleaned_text = re.sub(r'[^a-zA-Z\s]', '', raw_text)
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
        words = cleaned_text.lower().split()
 
        word_counts = Counter(words)
        df = pd.DataFrame(word_counts.items(), columns=['Word', 'Count'])
        df = df.sort_values(by='Count', ascending=False)
 
        df["Count"]=df["Count"].astype(int)
 
                
        
    
        
        
    elif book_title=="Lec":
        file_path = "/Users/dannysatterthwaite/Downloads/lecomnte - pocs - hw-6-5.rtf"
        
                    # read it in
        with open(file_path, 'r') as file:
            raw_text = file.read()
 
 
 
        # strip all the bad characters and new line characters.
        cleaned_text = re.sub(r'[^a-zA-Z\s]', '', raw_text)
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
        words = cleaned_text.lower().split()
 
        word_counts = Counter(words)
        df = pd.DataFrame(word_counts.items(), columns=['Word', 'Count'])
        df = df.sort_values(by='Count', ascending=False)
 
        df["Count"]=df["Count"].astype(int)
        
    
 
    
    display(df)
 
 
 
    count_once = 0
    count_twice = 0
    count_thrice = 0
 
    # Loop through the dataframe and increment the respective counter
    for count in df['Count']:
        if count == 1:
            count_once += 1
        elif count == 2:
            count_twice += 1
        elif count == 3:
            count_thrice += 1
 
    # Print the results
    print("The total number of words is: ", df["Count"].sum(), " and the number of unique words is: ", len(df))
    print(f"Unique Words that appear once: {count_once}")
    print(f"Unique Words that appear twice: {count_twice}")
    print(f"Unique Words that appear three times: {count_thrice}")
    print("\n-------\n")
    print("Thus our n_1^g estimate is: ", (count_once)/(len(df)))
    print("Thus our n_2^g estimate is: ", (count_twice)/(len(df)))
    print("Thus our n_3^g estimate is: ", (count_thrice)/(len(df)))
    print("\n-------------\n")
    
    # Innovation rate estimate
    rho_est = len(df)/df["Count"].sum()
    
    return rho_est
 
# Function to calculate theoretical and empirical values
def calculate_theoretical_and_empirical(rho_est):
    n_1 = 1 / (2 - rho_est)
    n_2 = n_1 * (1 - rho_est) / (1 + (1 - rho_est) * 2)
    n_3 = n_2 * (2 - rho_est) / (2 + (2 - rho_est) * 3)
    print("n_1^g should be based on the theoretical rho: ", n_1)
    print("n_2^g should be based on the theoretical rho: ", n_2)
    print("n_3^g should be based on the theoretical rho: ", n_3)
    print("-------------")
 
# Title list
title_list = ["u", "pp", "Lec"]
#title_list = ["u", "pp"]
 
for title in title_list:
    print(" \n\nthe book title is: ", title)
    print("--------")
    rho_est = read_word_counts(title)
    print(" the theoretical (est) rho is: ", rho_est, "\n")
    calculate_theoretical_and_empirical(rho_est)
 